[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import scipy.integrate as integrate
import scipy.optimize as optimize
import scipy.stats as stats

# 在不知道底层模型的情况下做回归

[视频时间戳](https://youtu.be/jReGEZXq4Ac?t=2450)

课程前半部分我们研究了线性回归，并做了一个关键假设：要估计参数，我们得先知道底层模型是线性的。如果去掉这个假设会怎样？

为了理解这一点，我们仍然看一个回归任务，但放松假设如下：我们知道底层模型是一个带加性噪声的多项式：
$$
y(i) = p^*(x(i)) + \epsilon(i),
$$
其中 $p^*$ 是一个未知的多项式，需要我们去估计。


In [ ]:
t = 0.5
sigma = 0.2
def model(x):
    return t*x**3 + sigma*np.random.randn()

In [ ]:
D = np.random.uniform(-2,2,25)
D = np.sort(D)
Y = [model(d) for d in D]

In [ ]:
plt.scatter(D,Y)
D_plot=  np.arange(-2,2.1,0.015)
plt.plot(D_plot,t*D_plot**3)

## 之前的方法哪里出了问题？

[视频时间戳](https://youtu.be/jReGEZXq4Ac?t=2578)

我们可以定义二次损失：
$$
J(p) = \frac{1}{m}\sum_{i=1}^m \left(y(i)-p(x(i)) \right)^2,
$$
并在所有多项式中最小化它。


In [ ]:
z = np.polyfit(D, Y,len(D)-1)
p = np.poly1d(z)

In [ ]:
plt.scatter(D,Y)
plt.plot(D_plot,t*D_plot**3)
plt.ylim(-4,4)
plt.plot(D_plot,p(D_plot))

很好，我们达到了 0 损失，但显然这个解看起来不对！

我们稍微偷个懒，看看如果使用正确次数的多项式解会是什么样：


In [ ]:
z = np.polyfit(D, Y,5)
p = np.poly1d(z)
plt.scatter(D,Y)
plt.plot(D_plot,t*D_plot**3)
plt.ylim(-4,4)
plt.plot(D_plot,p(D_plot))             

[视频时间戳](https://youtu.be/jReGEZXq4Ac?t=2720)

OK，这个看起来好多了，但我们作弊了！确实，你可以修改多项式拟合的次数，然后会发现很难在 3、4、5……这些次数之间做选择。

现在我们可以把问题形式化如下。给定点 $(x(i),y(i))$，我们需要做两件事：
 - 决定多项式 $p^*$ 的次数；
 - 次数一旦确定，就估计多项式的参数。

处理这个新问题的一种自然方式是：尝试所有可能的次数，对每个选择都做一次参数估计。但然后呢，我们需要决定选哪个次数。为此，我们把数据集分成训练集和验证集。用训练集对所有可能的次数估计多项式参数。为了决定选哪个次数，我们在验证集上计算得到的多项式的损失，选验证损失最小的那个。

看看这样是否可行？


In [ ]:
D_train = D[1::2]
Y_train = Y[1::2]
D_val = D[::2]
Y_val = Y[::2]
plt.scatter(D_train,Y_train)
plt.scatter(D_val, Y_val)

In [ ]:
def get_error(deg):
    val_error = np.zeros(deg)
    train_error = np.zeros(deg)
    for i in range(deg):
        z = np.polyfit(D_train, Y_train, i)
        p = np.poly1d(z)
        train_error[i] = (np.mean((p(D_train)-Y_train)**2))
        val_error[i] = (np.mean((p(D_val)-Y_val)**2))
    return train_error, val_error

In [ ]:
train_error, val_error = get_error(len(D_train)-1)
plt.figure(figsize=(14,7))
plt.plot(train_error, label='Training error')
plt.ylim(0, 3)
plt.plot(val_error, label='Validation error')
plt.xlabel("Degree")
plt.legend();

可以看到，训练集上的误差持续下降，直到多项式能够插值训练集的所有点时降到 0。

验证集上的误差先是随着训练误差一起下降，然后又开始上升。这是因为穿过了训练集所有点的多项式，现在漏掉了验证集的很多点，如下所示：


In [ ]:
z = np.polyfit(D_train, Y_train, len(D_train)-1)
p = np.poly1d(z)
plt.scatter(D_val, Y_val)
plt.scatter(D_train,Y_train)
plt.plot(D_plot,t*D_plot**3)
plt.ylim(-4,4)
plt.plot(D_plot,p(D_plot)) 

[视频时间戳](https://youtu.be/jReGEZXq4Ac?t=2935)

总结一下：
 - 次数太低时，我们的参数化模型表达力不够，无法刻画真实模型，导致训练集和验证集上的误差都很高。
 - 次数太高时，模型变得非常有表现力，开始真的去拟合数据集里的噪声，导致训练误差很低、验证误差很高。

为了稍微形式化地描述这一切，我们需要引入风险的概念。对估计器 $f:\mathbb{R}\to\mathbb{R}$，我们把风险定义为：
$$
\mathcal{R}(f) = \mathbb{E}\left[(f(X)-Y)^2\right],
$$
其中平均值是对 $(X,Y)$ 的随机性取的。在我们的例子里，我们用 $X\sim Unif[-2,2]$ 和 $Y = 0.5*X^3+\sigma \epsilon$ 来模拟真实模型，其中 $\epsilon\sim \mathcal{N}(0,1)$。

我们还把 $\mathcal{H}_k$ 定义为最高次数为 $k$ 的多项式集合。$\mathcal{H}_k$ 就是假设空间。我们用 $H_\infty$ 表示所有多项式的集合。

我们的目标是找到：
$$
p^* = \arg\min_{p\in \mathcal{H}_\infty}\mathcal{R}(p).
$$

在我们的例子里，可以算出这个风险：
\begin{eqnarray*}
\min_{p\in \mathcal{H}_\infty}\mathcal{R}(p) &=& \mathbb{E}\left[(Y-\mathbb{E}[Y|X])^2\right]\\
&=& \mathbb{E}\left[ (Y-0.5 X^3)^2\right]\\
&=& \sigma^2
\end{eqnarray*}

但在实践中，我们无法接触到定义 $(X,Y)$ 分布的真实底层模型，所以无法计算定义风险的那个平均值。

因此，我们定义经验风险：
$$
\hat{\mathcal{R}}(p) = \frac{1}{m}\sum_{i=1}^m \left(y(i)-p((x(i))\right)^2,
$$
它是真实风险的一个近似。
更准确地说，我们定义了两个不同的经验风险：$\hat{\mathcal{R}}_{train}(p)$ 是对训练集取平均，$\hat{\mathcal{R}}_{val}(p)$ 是对验证集取平均。

现在我们定义在训练集上最小化经验风险的、次数至多为 $k$ 的多项式：
$$
\hat{p}_k = \arg\min_{p\in \mathcal{H}_k}\hat{\mathcal{R}}_{train}(p).
$$

上面的训练误差就是 $\hat{\mathcal{R}}_{train}(\hat{p}_k)$，验证误差就是 $\hat{\mathcal{R}}_{val}(\hat{p}_k)$。

由于验证集里的数据点没有参与多项式拟合，所以 $\hat{\mathcal{R}}_{val}(\hat{p}_k)\approx \mathcal{R}(\hat{p}_k)$。

不幸的是，我们真正想算的是
$$
p^*_k = \arg\min_{p\in \mathcal{H}_k}\mathcal{R}(p).
$$
注意 $\mathcal{R}(p^*_k) \downarrow \mathcal{R}(p^*)$（当 $k\to \infty$ 时）。
不幸的是，上面的实验告诉我们 $p^*_k\neq \hat{p}_k$，特别是 $k$ 很大时。

[视频时间戳](https://youtu.be/jReGEZXq4Ac?t=3280)

无论如何，我们都可以把估计器的风险分解成下面这些非负项：
$$
\mathcal{R}(\hat{p}_k) = \underbrace{\mathcal{R}(\hat{p}_k)-\mathcal{R}(p^*_k)}_{(1)} + \underbrace{\mathcal{R}(p^*_k)-\mathcal{R}(p^*)}_{(2)}+\mathcal{R}(p^*).
$$

第一项叫做**估计误差**，第二项叫做**逼近误差**，最后一项 $\mathcal{R}(p^*)$ 是真实风险。

显然，随着 $k\to \infty$，模型越来越有表达力，逼近误差 (2) 趋于 0。在我们的例子里，$k\geq 3$ 时逼近误差就是 0。但遗憾的是估计误差 (1) 会随着 $k$ 增长。回到上面的实验发现：$k$ 较小时，$\mathcal{R}(\hat{p}_k)$ 高是因为逼近误差 (2) 高；$k$ 较大时，$\mathcal{R}(\hat{p}_k)$ 高是因为估计误差 (1) 高。实践中，我们取这条曲线的最小值作为 $\mathcal{R}(p^*)$ 的估计。


In [ ]:
def get_risk(deg):
    risk = np.zeros(deg)
    for i in range(deg):
        z = np.polyfit(D_train, Y_train, i)
        p = np.poly1d(z)
        fun_int= lambda e,x : (p(x)-t*x**3 - sigma*e)**2*stats.norm.pdf(e)/4
        risk[i] = integrate.dblquad(fun_int,-2,2,lambda x: -10, lambda x: 10)[0]
    return risk

In [ ]:
risk = get_risk(len(D_train)-1)

In [ ]:
def opti_risk(x):
    if int(x) == 0:
        return t**2*2**6/7+sigma**2
    elif int(x) == 1:
        return t**2*2**6/7+sigma**2-t**2*2**6/5**2*3
    elif int(x) == 2:
        pass# left as an exercise!
    else:
        return sigma**2

In [ ]:
deg = [0,1,3,4,5,6,7,8,9,10]
plt.figure(figsize=(14,7))
plt.ylim(0, 3)
plt.plot(deg, [risk[int(d)]-opti_risk(d) for d in deg], color='red',label='estimation error')
plt.plot(deg, [opti_risk(d)-sigma**2 for d in deg],color='green',label='approximation error')
plt.plot(val_error, label='empirical validation error')
plt.xlabel("Degree")
plt.legend();

In [ ]:
plt.figure(figsize=(14,7))
plt.ylim(0, 3)
plt.plot(risk, color='red',label='risk of esimator $\mathcal{R}(\hat{p}_k)$')
plt.plot(val_error, label='empirical validation error $\hat{\mathcal{R}}_{val}(\hat{p}_k)$')
plt.xlabel("Degree")
plt.legend();

[![Dataflowr](https://raw.githubusercontent.com/dataflowr/website/master/_assets/dataflowr_logo.png)](https://dataflowr.github.io/website/)